In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt

# ── 1. Load data ─────────────────────────────────────────────────
df_bind = pd.read_excel('../data/single_metal_atoms_on_graphene_binding_energy_and_diffusion_barrier.xlsx')
df_bind_g = df_bind.groupby('Metal')[['Binding', 'Diffusion']].mean().reset_index()

df_aff = pd.read_excel("../data/supported_metal_M_oxygen_affinity_QMO_and_support_metal_affinity_QMM_prime.xlsx")
df_aff = df_aff.rename(columns={df_aff.columns[1]: 'QMO'})

df_nps = pd.read_excel('../data/all_NPs.xlsx')
df_nps = df_nps.rename(columns={'反应后MOF表面是否有纳米粒子': 'ExternalNP'})
df_nps['MOF'] = df_nps['MOF'].ffill()

# Define your renaming dictionary
mof_rename_dict = {
}

# Apply the replacement
df_nps["MOF"] = df_nps["MOF"].replace(mof_rename_dict)

df_mof = pd.read_csv('../data/MOF_factor.csv').reset_index(drop = True)
# Optional: check for consistency with df_mof
for i in df_nps["MOF"]:
    assert i in list(df_mof["MOF"]), f"{i} not in df_mof"
# ── 2. Merge all descriptors ─────────────────────────────────────
df = (
    df_nps
    .merge(df_mof, on='MOF', how='left')
    .merge(df_bind_g.rename(columns={'Binding': 'BindingEnergy', 'Diffusion': 'DiffusionBarrier'}),
           left_on='M', right_on='Metal', how='left')
    .merge(df_aff[['Metal', 'QMO']], left_on='M', right_on='Metal', how='left')
    .drop(columns=['Metal', 'Metal_aff'], errors='ignore')
)

df = df.dropna(subset=['BindingEnergy', 'DiffusionBarrier', 'QMO'])
df = df.drop(columns=['Metal_x', 'Metal_y'], errors='ignore')

noble_set = {'Au', 'Ag', 'Pt', 'Pd', 'Ir', 'Rh', 'Ru'}
df['Noble'] = df['M'].apply(lambda x: 1 if x in noble_set else 0)

In [2]:
from sklearn.base import clone
import os
from sklearn.neural_network import MLPClassifier

models = {
    "MLP": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(10,4), activation="tanh",
                              max_iter=100000, random_state=42, tol=1e-6))
    ]),
}
mask = np.array([not i in ["UiO-66-I", "UiO-66-F"] for i in df["MOF"].values])

In [3]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix

# --------------------------------------------------
# 1. define feature sets
# --------------------------------------------------
feats_without_f37 = ['DiffusionBarrier', 'QMO', 'Noble', 'BindingEnergy']
feats_with_f37    = ['DiffusionBarrier', 'QMO', 'Noble', 'Factor37']
target_col = 'ExternalNP'

# IMPORTANT:
# to compare fairly, use only rows that have all features needed by BOTH models
needed_cols = list(set(feats_without_f37 + feats_with_f37 + [target_col, 'MOF', 'M']))
df_cmp = df.dropna(subset=needed_cols).copy()

# external test = UiO-66-I and UiO-66-F
test_mofs = ["UiO-66-I", "UiO-66-F"]
mask = ~df_cmp["MOF"].isin(test_mofs)   # True = train, False = external test

train_df = df_cmp[mask].copy()
test_df  = df_cmp[~mask].copy()

print("Train size:", len(train_df))
print("External test size:", len(test_df))
print("External test MOFs:", test_df["MOF"].value_counts().to_dict())

# --------------------------------------------------
# 2. prepare train/test data
# --------------------------------------------------
X_train_without = train_df[feats_without_f37].copy()
X_test_without  = test_df[feats_without_f37].copy()

X_train_with = train_df[feats_with_f37].copy()
X_test_with  = test_df[feats_with_f37].copy()

y_train = train_df[target_col].astype(int).copy()
y_test  = test_df[target_col].astype(int).copy()

# --------------------------------------------------
# 3. fit and evaluate both models
# --------------------------------------------------
results = {}

for model_name, Xtr, Xte in [
    ("BindingEnergy model", X_train_without, X_test_without),
    ("Factor37 model", X_train_with, X_test_with),
]:
    model = clone(models["MLP"])
    model.fit(Xtr, y_train)

    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    res = {
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) == 2 else np.nan,
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "y_pred": y_pred,
        "y_prob": y_prob,
        "model": model,
    }
    results[model_name] = res

# --------------------------------------------------
# 4. print summary
# --------------------------------------------------
for name, res in results.items():
    print(f"\n{name}")
    print(f"  External test accuracy:          {res['accuracy']:.3f}")
    print(f"  External test balanced accuracy: {res['balanced_accuracy']:.3f}")
    print(f"  External test ROC-AUC:           {res['roc_auc']:.3f}")
    print(f"  Confusion matrix:\n{res['confusion_matrix']}")

# --------------------------------------------------
# 5. simple comparison
# --------------------------------------------------
acc_bind = results["BindingEnergy model"]["accuracy"]
acc_f37  = results["Factor37 model"]["accuracy"]

print("\nComparison on external test set")
print(f"BindingEnergy model accuracy: {acc_bind:.3f}")
print(f"Factor37 model accuracy:      {acc_f37:.3f}")
print(f"Delta (Factor37 - Binding):   {acc_f37 - acc_bind:+.3f}")

Train size: 110
External test size: 20
External test MOFs: {'UiO-66-F': 10, 'UiO-66-I': 10}

BindingEnergy model
  External test accuracy:          0.750
  External test balanced accuracy: 0.775
  External test ROC-AUC:           0.885
  Confusion matrix:
[[9 4]
 [1 6]]

Factor37 model
  External test accuracy:          0.850
  External test balanced accuracy: 0.819
  External test ROC-AUC:           0.890
  Confusion matrix:
[[12  1]
 [ 2  5]]

Comparison on external test set
BindingEnergy model accuracy: 0.750
Factor37 model accuracy:      0.850
Delta (Factor37 - Binding):   +0.100
